# Kafka consumer

In [ ]:
from confluent_kafka import Consumer

conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'test',
    'auto.offset.reset': 'earliest',
}
topic1 = 'traffic-object-raw'
topic2 = 'weather-raw'
consumer = Consumer(conf)
# consumer.subscribe(['weather-raw'])
consumer.subscribe([topic2])


try:
    while True:
        msg = consumer.poll(timeout=1.0)  # Polls for up to 1 second
        if msg is None:
            continue  # No message, go back and poll again
        if msg.error():
            print(f"Consumer error: {msg.error()}")
            continue
        print(f"Received message: {msg.value().decode('utf-8')}")
except KeyboardInterrupt:
    pass
finally:
    consumer.close()


Received message: {"image_shape": [720, 1280], "total": 3, "objects": [{"class_object": "person", "coordinates": [100, 150, 200, 300], "confidence": 0.92, "class_id": 0, "classname": "person"}, {"class_object": "bicycle", "coordinates": [250, 400, 300, 500], "confidence": 0.87, "class_id": 1, "classname": "bicycle"}, {"class_object": "bus", "coordinates": [600, 700, 650, 750], "confidence": 0.95, "class_id": 2, "classname": "bus"}], "timestamp": "2025-05-02 13:13:56", "cam_id": "56de42f611f398ec0c481289", "img": "datalake/raw/traffic/56de42f611f398ec0c481289/2025-05-02/13-13-56.jpg"}
Received message: {"image_shape": [720, 1280], "total": 3, "objects": [{"class_object": "person", "coordinates": [100, 150, 200, 300], "confidence": 0.92, "class_id": 0, "classname": "person"}, {"class_object": "bicycle", "coordinates": [250, 400, 300, 500], "confidence": 0.87, "class_id": 1, "classname": "bicycle"}, {"class_object": "bus", "coordinates": [600, 700, 650, 750], "confidence": 0.95, "class_id

In [17]:
from confluent_kafka import Consumer, TopicPartition
from datetime import datetime, timedelta

topic = 'weather-raw'
interval = 15 # last 15 min

conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'test',
    'enable.auto.commit': False,
    'auto.offset.reset': 'earliest'  # Only used if no committed offsets exist
}

consumer = Consumer(conf)
metadata = consumer.list_topics(topic)
partitions = metadata.topics[topic].partitions.keys()
# topic_partitions = [TopicPartition(topic, p) for p in partitions]

# timestamp for last 15 mins
timestamp_ms = int((datetime.now() - timedelta(minutes=interval)).timestamp() * 1000)
timestamp_partitions = [TopicPartition(topic, p, timestamp_ms) for p in partitions]
offsets = consumer.offsets_for_times(timestamp_partitions, timeout=10.0)

# Assign and seek to the correct offsets
valid_offsets = []
for tp in offsets:
    if tp.offset != -1:  # Offset -1 means no data available for that timestamp
        valid_offsets.append(TopicPartition(tp.topic, tp.partition, tp.offset))

consumer.assign(valid_offsets)

# Start consuming
while True:
    msg = consumer.poll(1.0)  # Waits up to 1 second for new messages
    if msg is None:
        break  # Stop when nothing is returned
    if msg.error():
        print(f"Error: {msg.error()}")
        continue
    print(msg.value().decode())


# Google cloud Storage